# M2→M3: From Symbolic Gradients to a Trained MLP

Three acts, each building on what the last one proved:

1. **Act 1 — one neuron, by hand.** Derive `dL/dw` and `dL/db` for a single sigmoid
   neuron with sympy, lambdify into numpy, and run gradient descent using *only*
   those lambdified functions — no pipeline model classes yet.
2. **Act 2 — the bridge.** Derive the *same kind* of gradients for the full
   7→16→2 MLP architecture Act 3 trains — symbolically, matrix by matrix — then
   check numerically that `NumpyModel.backward()` (the pipeline's own backprop)
   produces the identical numbers. This is the proof that the library's autodiff
   isn't a black box: it's the same chain rule, just implemented once instead of
   re-derived per model.
3. **Act 3 — train for real.** With `backward()` trusted, train the same MLP on
   Titanic at full scale via `NumpyOptimizer` + `Adam` + the pipeline's `TrainLoop`.

## Act 1: A Single Neuron's Gradient, Derived by Hand

No `NumpyModel`, no `NumpyOptimizer` — just sympy, `lambdify`, and a manual update
loop. The point is to see the gradient formula *before* trusting a library to
compute it for you.

In [ ]:
import sympy as sp
import numpy as np

# Single neuron: y_hat = sigmoid(w*x + b)
w, b, x, t = sp.symbols('w b x t')
y_hat = 1 / (1 + sp.exp(-(w * x + b)))

# Binary cross-entropy loss: L = -[t*log(y_hat) + (1-t)*log(1-y_hat)]
L = -(t * sp.log(y_hat) + (1 - t) * sp.log(1 - y_hat))

# Symbolic gradients
dL_dw = sp.simplify(sp.diff(L, w))
dL_db = sp.simplify(sp.diff(L, b))
print('dL/dw =', dL_dw)

dL/dw = x*(-t*exp(b + w*x) - t + exp(b + w*x))/(exp(b + w*x) + 1)
dL/db = (-t*exp(b + w*x) - t + exp(b + w*x))/(exp(b + w*x) + 1)


In [2]:
print('dL/db =', dL_db)

NameError: name 'dL_db' is not defined

In [2]:
# Lambdify: turn the symbolic expressions into fast numpy-callable functions
grad_w_fn = sp.lambdify([w, b, x, t], dL_dw, 'numpy')
grad_b_fn = sp.lambdify([w, b, x, t], dL_db, 'numpy')

# Sanity check against one point
w_val, b_val, x_val, t_val = 0.5, 0.0, 1.0, 1.0
print(f'grad_w = {grad_w_fn(w_val, b_val, x_val, t_val):.6f}')
print(f'grad_b = {grad_b_fn(w_val, b_val, x_val, t_val):.6f}')

grad_w = -0.377541
grad_b = -0.377541


In [ ]:
# Manual gradient descent — using ONLY the lambdified functions above.
# Task: classify x >= 0 as class 1, x < 0 as class 0 (sigmoid can represent this;
# a plain linear regressor could not, since sigmoid squashes to (0, 1)).
X = np.array([-3.0, -2.0, -1.0, 1.0, 2.0, 3.0])
T = np.array([0.0, 0.0, 0.0, 1.0, 1.0, 1.0])

w_val, b_val = 0.1, 0.0
lr = 0.5
bce_losses = []

for epoch in range(200):
    grad_w_sum, grad_b_sum, loss_sum = 0.0, 0.0, 0.0
    for x_i, t_i in zip(X, T):
        grad_w_sum += grad_w_fn(w_val, b_val, x_i, t_i)
        grad_b_sum += grad_b_fn(w_val, b_val, x_i, t_i)
        y_hat_i = 1 / (1 + np.exp(-(w_val * x_i + b_val)))
        loss_sum += -(t_i * np.log(y_hat_i + 1e-12) + (1 - t_i) * np.log(1 - y_hat_i + 1e-12))

    n = len(X)
    w_val -= lr * grad_w_sum / n
    b_val -= lr * grad_b_sum / n
    bce_losses.append(loss_sum / n)

print(f'Final loss: {bce_losses[-1]:.6f}')
print(f'Learned w: {w_val:.4f}, b: {b_val:.4f}')
preds = (1 / (1 + np.exp(-(w_val * X + b_val)))) >= 0.5
print(f'Predictions correct: {np.array_equal(preds, T.astype(bool))}')

In [ ]:
# Manual gradient descent — single neuron learning y = 2x
from pipeline.adapters.numpy_adapter import NumpyModel, NumpyOptimizer
from pipeline.training.optimizers import SGD
from pipeline.protocols import Parameter

# Initialize parameters
W_data = np.array([[0.5]])
b_data = np.array([0.0])
model = NumpyModel([(W_data, b_data)], activation='relu')
opt = NumpyOptimizer(model.parameters(), SGD(lr=0.1))

# Training data: y = 2x
X = np.array([[1.0], [2.0], [3.0], [4.0]])
y = np.array([[2.0], [4.0], [6.0], [8.0]])

mse_losses = []
for epoch in range(50):
    epoch_loss = 0
    for i in range(len(X)):
        x_i = X[i:i+1]
        y_i = y[i:i+1]
        pred = model.forward(x_i)
        loss_val = float(np.mean((pred - y_i)**2))
        epoch_loss += loss_val

        # Manual gradient: dL/dpred = 2*(pred - y_i)
        dL_dpred = 2 * (pred - y_i)
        model.backward(dL_dpred)

        opt.step()
        opt.zero_grad()

    mse_losses.append(epoch_loss / len(X))

print(f'Final loss: {mse_losses[-1]:.6f}')
W_learned = list(model.parameters())[0].data[0, 0]
b_learned = list(model.parameters())[1].data[0]
print(f'Learned W: {W_learned:.4f} (target: 2.0)')
print(f'Learned b: {b_learned:.4f} (target: 0.0)')

In [ ]:
# Plot the BCE loss curve from the hand-derived-gradient run (cell above the y=2x demo)
import matplotlib.pyplot as plt
plt.plot(bce_losses)
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('Single Neuron: Gradient Descent from Hand-Derived Gradients')
plt.show()

**What we just did:** derived `dL/dw` and `dL/db` by hand (sympy), converted them to
numpy (`lambdify`), and ran gradient descent using *nothing but those two functions*
— no model class, no optimizer object. The neuron learned to separate the two classes.

**The gap:** a real network has hundreds of parameters. Deriving `dL/dW1`, `dL/db1`,
`dL/dW2`, `dL/db2` by hand the same way, one scalar at a time, would not scale. That's
what `NumpyModel.backward()` is *for* — but before trusting it, let's check it actually
computes what sympy says it should.

## Act 2: The Bridge — Deriving the Full MLP's Gradients Symbolically

Same architecture Act 3 trains: **7 inputs → 16 hidden (ReLU) → 2 outputs (softmax +
cross-entropy)**. We build the forward pass in sympy as matrix expressions, differentiate
w.r.t. every entry of `W1`, `b1`, `W2`, `b2`, lambdify the result, then evaluate on one
random sample and compare against `NumpyModel.forward()` + `CrossEntropyLoss.backward()`
+ `model.backward()` — the pipeline's actual code path.

If the numbers match, `backward()` is trustworthy at any scale. If they don't, we've
found a bug before training a single epoch.

In [6]:
import time
from sympy import Symbol, Matrix, Max, exp as sp_exp, log as sp_log

N_IN, N_HID, N_OUT = 7, 16, 2  # same shape NumpyModel([(W1,b1), (W2,b2)]) uses in Act 3

# Symbolic parameters — one Symbol per matrix entry, matching Parameter.data shapes
x_syms = list(sp.symbols(f'x0:{N_IN}'))
W1_syms = [[Symbol(f'W1_{i}_{j}') for j in range(N_IN)] for i in range(N_HID)]
b1_syms = [Symbol(f'b1_{i}') for i in range(N_HID)]
W2_syms = [[Symbol(f'W2_{i}_{j}') for j in range(N_HID)] for i in range(N_OUT)]
b2_syms = [Symbol(f'b2_{i}') for i in range(N_OUT)]
t0_sym, t1_sym = sp.symbols('t0 t1')  # one-hot target for the 2 output classes

W1_sym = Matrix(W1_syms)
b1_sym = Matrix(N_HID, 1, lambda i, j: b1_syms[i])
W2_sym = Matrix(W2_syms)
b2_sym = Matrix(N_OUT, 1, lambda i, j: b2_syms[i])
x_vec = Matrix(N_IN, 1, lambda i, j: x_syms[i])

# Forward pass, matching NumpyModel.forward(): z = x @ W.T + b, ReLU on hidden layer only
z1 = W1_sym * x_vec + b1_sym
a1 = z1.applyfunc(lambda e: Max(e, 0))          # ReLU
z2 = W2_sym * a1 + b2_sym                        # raw logits (no activation on last layer)

# Softmax + cross-entropy, matching CrossEntropyLoss.forward()
exp0, exp1 = sp_exp(z2[0, 0]), sp_exp(z2[1, 0])
p0, p1 = exp0 / (exp0 + exp1), exp1 / (exp0 + exp1)
L = -(t0_sym * sp_log(p0) + t1_sym * sp_log(p1))

n_params = N_HID * N_IN + N_HID + N_OUT * N_HID + N_OUT
print(f'Forward graph built: {N_IN}->{N_HID}(ReLU)->{N_OUT}(softmax+CE), {n_params} scalar parameters')

Forward graph built: 7->16(ReLU)->2(softmax+CE), 162 scalar parameters


In [7]:
# Differentiate w.r.t. every parameter entry, then lambdify.
# Takes ~5s (diff) + ~15s (lambdify) — one-time cost for 162 symbolic gradients.
all_param_syms = (
    [s for row in W1_syms for s in row] + b1_syms
    + [s for row in W2_syms for s in row] + b2_syms
)

t_start = time.time()
grad_exprs = [sp.diff(L, s) for s in all_param_syms]
print(f'Differentiated {len(grad_exprs)} params in {time.time() - t_start:.1f}s')

t_start = time.time()
grad_fn = sp.lambdify(all_param_syms + x_syms + [t0_sym, t1_sym], grad_exprs, 'numpy')
print(f'Lambdified in {time.time() - t_start:.1f}s')

Differentiated 162 params in 4.4s


Lambdified in 16.1s


In [8]:
from pipeline.adapters.numpy_adapter import NumpyModel
from pipeline.training.losses import CrossEntropyLoss

# One random sample, one random parameter init — same shapes NumpyModel expects.
rng = np.random.default_rng(0)
W1_val = rng.standard_normal((N_HID, N_IN)) * 0.5
b1_val = rng.standard_normal(N_HID) * 0.1
W2_val = rng.standard_normal((N_OUT, N_HID)) * 0.5
b2_val = rng.standard_normal(N_OUT) * 0.1
x_val = rng.standard_normal(N_IN)
target_class = 1  # one-hot: t0=0, t1=1

# --- sympy path: evaluate the lambdified symbolic gradients ---
flat_vals = (
    list(W1_val.flatten()) + list(b1_val) + list(W2_val.flatten()) + list(b2_val)
    + list(x_val) + [0.0, 1.0]
)
sympy_grads = np.array(grad_fn(*flat_vals), dtype=np.float64)
n1 = N_HID * N_IN
sympy_dW1 = sympy_grads[:n1].reshape(N_HID, N_IN)
sympy_db1 = sympy_grads[n1:n1 + N_HID]
n2 = n1 + N_HID + N_OUT * N_HID
sympy_dW2 = sympy_grads[n1 + N_HID:n2].reshape(N_OUT, N_HID)
sympy_db2 = sympy_grads[n2:]

# --- pipeline path: NumpyModel.forward() + CrossEntropyLoss + model.backward() ---
model = NumpyModel([(W1_val.copy(), b1_val.copy()), (W2_val.copy(), b2_val.copy())])
loss_fn = CrossEntropyLoss(model=model)

logits = model.forward(x_val.reshape(1, -1))
loss = loss_fn(logits, np.array([target_class]))
loss.backward()

params = list(model.parameters())  # [W1, b1, W2, b2] Parameter objects
pipeline_dW1, pipeline_db1, pipeline_dW2, pipeline_db2 = (p.grad for p in params)

# --- compare ---
checks = [
    ('dL/dW1', sympy_dW1, pipeline_dW1),
    ('dL/db1', sympy_db1, pipeline_db1),
    ('dL/dW2', sympy_dW2, pipeline_dW2),
    ('dL/db2', sympy_db2, pipeline_db2),
]
for name, sympy_val, pipeline_val in checks:
    match = np.allclose(sympy_val, pipeline_val, atol=1e-8)
    max_diff = np.max(np.abs(sympy_val - pipeline_val))
    print(f'{name:8s}  match={match}  max_abs_diff={max_diff:.2e}')

dL/dW1    match=True  max_abs_diff=4.44e-16
dL/db1    match=True  max_abs_diff=2.22e-16
dL/dW2    match=True  max_abs_diff=4.44e-16
dL/db2    match=True  max_abs_diff=1.11e-16


**What we just proved:** the same chain rule from Act 1 — differentiate the loss
w.r.t. each parameter — still holds for a 162-parameter network. `NumpyModel.backward()`
gets identical gradients to sympy's symbolic derivation (max abs diff ~1e-16, i.e.
float rounding, not a discrepancy). We derived it once here to verify it; the library
computes it every batch so we never have to re-derive it by hand.

**Now:** train this architecture on the real Titanic dataset — same math, real data,
many epochs.

## Act 3: Train the Verified Architecture on Titanic

Same 7→16→2 shape, same `NumpyModel` + `CrossEntropyLoss` code path Act 2 just checked
against sympy — now with real data, `NumpyOptimizer` + `Adam`, and the pipeline's
`TrainLoop` running 100 epochs instead of one manual step.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, LabelEncoder

from pipeline.config import Config
from pipeline.data.csv_source import CsvDataSource
from pipeline.data.split import train_test_split
from pipeline.evaluation.metrics import Metrics, accuracy
from pipeline.hooks.progress import ProgressHook
from pipeline.pipeline import BasePipeline, PipelineState
from pipeline.adapters.numpy_adapter import NumpyModel, NumpyOptimizer
from pipeline.training.losses import CrossEntropyLoss
from pipeline.training.optimizers import Adam

# Fetch and preprocess Titanic
titanic = fetch_openml('titanic', version=1, as_frame=True, parser='auto')
df = titanic.data.copy()
df['survived'] = titanic.target.astype(int)
cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']
df = df[cols].copy()
df['sex'] = LabelEncoder().fit_transform(df['sex'])
df['embarked'] = LabelEncoder().fit_transform(df['embarked'].astype(str))
for c in ['age', 'fare']:
    df[c] = df[c].fillna(df[c].median())
df['embarked'] = df['embarked'].fillna(0)

# Scale features
scaler = StandardScaler()
feature_cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
X = scaler.fit_transform(df[feature_cols].astype(float))
scaled_df = pd.DataFrame(X, columns=feature_cols)
scaled_df['survived'] = df['survived'].astype(int)

import tempfile, os
_tmpdir = tempfile.mkdtemp()
csv_path = os.path.join(_tmpdir, 'titanic.csv')
scaled_df.to_csv(csv_path, index=False)
print(f'Data: {scaled_df.shape[0]} rows, {len(feature_cols)} features')

Data: 1309 rows, 7 features


In [ ]:
config = Config(
    batch_size=32, num_epochs=100, train_ratio=0.8, learning_rate=0.01,
    seed=42, output_dir=f'{_tmpdir}/output', task_name='titanic-numpy-mlp',
)


class TitanicNumpyMLP(BasePipeline):
    def load_data(self, state: PipelineState) -> None:
        source = CsvDataSource(
            csv_path, batch_size=config.batch_size,
            target_column='survived', shuffle=True, seed=config.seed,
        )
        train, val = train_test_split(source, train_ratio=config.train_ratio)
        state.data_stream = train
        state.val_data_stream = val

    def build_model(self, state: PipelineState) -> None:
        rng = np.random.default_rng(config.seed)
        # 7 input features -> 16 hidden -> 2 output classes
        W1 = rng.standard_normal((16, 7)) * np.sqrt(2.0 / 7)
        b1 = np.zeros(16)
        W2 = rng.standard_normal((2, 16)) * np.sqrt(2.0 / 16)
        b2 = np.zeros(2)

        state.model = NumpyModel([(W1, b1), (W2, b2)], activation='relu')
        state.loss_fn = CrossEntropyLoss(model=state.model)
        state.optimizer = NumpyOptimizer(
            state.model.parameters(), Adam(lr=config.learning_rate),
        )
        state.metrics = Metrics(accuracy=accuracy)

    def evaluate(self, state: PipelineState) -> None:
        state.model.eval_mode()
        preds, trues = [], []
        for batch in state.val_data_stream:
            logits = np.asarray(state.model.forward(batch.inputs))
            preds.append(np.argmax(logits, axis=1))
            trues.append(np.asarray(batch.targets))
        y_pred = np.concatenate(preds)
        y_true = np.concatenate(trues).astype(np.int64)
        state.metrics.compute(y_true, y_pred)

    def export(self, state: PipelineState) -> None:
        state.predictions = np.array([0])

In [ ]:
pipeline = TitanicNumpyMLP(config)
pipeline.add_hook(ProgressHook())
state = pipeline.run('train')

print(f'Accuracy:    {state.metrics["accuracy"]:.4f}')
print(f'Final loss:  {state.history["loss"][-1]:.4f}')
print(f'Epochs:      {len(state.history["loss"])}')


Epoch 1:   0%|          | 0/1 [00:00<?, ?it/s]


Epoch 1: 100%|██████████| 1/1 [00:00<00:00, 1940.01it/s, loss=0.7780]


Epoch 1: 100%|██████████| 1/1 [00:00<00:00, 871.27it/s, loss=0.7780] 


Epoch 2:   0%|          | 0/1 [00:00<?, ?it/s]


Epoch 2: 100%|██████████| 1/1 [00:00<00:00, 1314.42it/s, loss=0.7291]

...


Epoch 99: 100%|██████████| 1/1 [00:00<00:00, 742.75it/s, loss=0.3986] 


Epoch 100:   0%|          | 0/1 [00:00<?, ?it/s]


Epoch 100: 100%|██████████| 1/1 [00:00<00:00, 1494.76it/s, loss=0.3984]


Epoch 100: 100%|██████████| 1/1 [00:00<00:00, 908.84it/s, loss=0.3984] 

Accuracy:    0.7672
Final loss:  0.3984
Epochs:      100


## Compare: M1 (sklearn) vs Act 3 (numpy MLP)

| | M1 sklearn | Act 3 numpy MLP |
|---|---|---|
| Model | LogisticRegression | 2-layer MLP (7→16→2) |
| Training | `.fit()` one call | 100 epochs per-batch GD |
| Optimizer | LBFGS (internal) | Adam (lr=0.01) |
| Loss | log-loss (internal) | CrossEntropyLoss |
| Parameters | None exposed | 4 Parameters (W1,b1,W2,b2) |
| Gradient | sklearn internal | `model.backward()` — verified against sympy in Act 2 |

**Key insight:** the same pipeline structure ran both. The sklearn path overrode
`train()`; the numpy path used the default `TrainLoop`. And unlike sklearn's opaque
internal solver, the numpy path's gradient computation was checked by hand in Act 2 —
you know exactly what `backward()` computes, because you derived it yourself first.